# Core dynamic-programming verification

This notebook turns the central BayesBreak Gaussian workflow into an inspectable verification report. It checks public interfaces, dynamic-programming invariants, posterior normalization, MAP reconstruction, prediction behavior, prior sensitivity, edge cases, and small-scale runtime trends.

The synthetic boundaries are known because this is a controlled software check. Agreement on these data is **not** a universal accuracy claim. All generated tables and figures are written to `results/notebook_verification/core_dp/`.

In [ ]:
from __future__ import annotations

import inspect
import json
import platform
import sys
import time
import traceback
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone

import bayesbreak
from bayesbreak import BayesBreakGaussian, run_dp_diagnostics, run_prior_sensitivity

SEED = 20260809
RNG = np.random.default_rng(SEED)
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
OUTPUT_DIR = ROOT / "results" / "notebook_verification" / "core_dp"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

ENVIRONMENT = {
    "python": platform.python_version(),
    "bayesbreak": bayesbreak.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": version("matplotlib"),
    "platform": platform.platform(),
    "seed": SEED,
}
print(json.dumps(ENVIRONMENT, indent=2))
print(f"Report directory: {OUTPUT_DIR.relative_to(ROOT)}")

## 1–3. Configure, load, and create deterministic fixtures

The environment cell fixes the random seed, records dependency versions, and creates a dedicated output directory. The fixture below has three Gaussian regimes with true boundaries at indices 35 and 70. A shuffled negative control preserves the marginal distribution while removing the ordered regime structure.

In [ ]:
segment_lengths = [35, 35, 40]
segment_means = [0.0, 2.25, -1.25]
true_boundaries = np.cumsum([0, *segment_lengths]).tolist()
latent_signal = np.repeat(segment_means, segment_lengths)
observations = latent_signal + RNG.normal(0.0, 0.45, latent_signal.size)
coordinates = np.arange(observations.size, dtype=float).reshape(-1, 1)
shuffled_observations = RNG.permutation(observations)

assert true_boundaries == [0, 35, 70, 110]
assert observations.shape == (110,)
assert np.array_equal(np.sort(observations), np.sort(shuffled_observations))

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.scatter(coordinates[:, 0], observations, s=15, color="#466365", alpha=0.72, label="observed")
ax.plot(coordinates[:, 0], latent_signal, color="#D1495B", linewidth=2.2, label="latent mean")
for boundary in true_boundaries[1:-1]:
    ax.axvline(boundary, color="#D1495B", linestyle="--", linewidth=1)
ax.set(xlabel="observation index", ylabel="response", title="Deterministic three-regime fixture")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fixture.png", bbox_inches="tight")
plt.show()

## 4. Inspect components and public interfaces

Before fitting, inspect constructor and method signatures directly. This catches accidental API drift while keeping the notebook tied to public objects rather than implementation internals.

In [ ]:
interface_rows = []
for name, callable_object in {
    "BayesBreakGaussian": BayesBreakGaussian,
    "fit": BayesBreakGaussian.fit,
    "predict": BayesBreakGaussian.predict,
    "transform": BayesBreakGaussian.transform,
    "score": BayesBreakGaussian.score,
    "run_dp_diagnostics": run_dp_diagnostics,
    "run_prior_sensitivity": run_prior_sensitivity,
}.items():
    docstring = inspect.getdoc(callable_object) or ""
    interface_rows.append(
        {
            "component": name,
            "signature": str(inspect.signature(callable_object)),
            "summary": docstring.splitlines()[0] if docstring else "",
        }
    )
interface_frame = pd.DataFrame(interface_rows)
display(interface_frame)
assert {"fit", "predict", "transform", "score"}.issubset(interface_frame["component"])
assert "k_max" in interface_frame.loc[interface_frame.component == "BayesBreakGaussian", "signature"].iloc[0]

## 5–6. Run checks and capture structured results

`record_check` keeps pass/fail/error status, expected and observed values, execution time, and tracebacks. The estimator is fitted once, then the notebook verifies posterior normalization, evidence-table shape, admissibility, MAP reconstruction, predictions, segment labels, score, and sklearn cloning.

In [ ]:
verification_records = []

def record_check(component, check, function, expected):
    started = time.perf_counter()
    try:
        actual = function()
        passed = bool(actual) if isinstance(actual, (bool, np.bool_)) else actual == expected
        verification_records.append(
            {
                "component": component,
                "check": check,
                "status": "pass" if passed else "fail",
                "expected": repr(expected),
                "actual": repr(actual),
                "elapsed_ms": 1000 * (time.perf_counter() - started),
                "traceback": "",
            }
        )
    except Exception as error:
        verification_records.append(
            {
                "component": component,
                "check": check,
                "status": "error",
                "expected": repr(expected),
                "actual": f"{type(error).__name__}: {error}",
                "elapsed_ms": 1000 * (time.perf_counter() - started),
                "traceback": traceback.format_exc(),
            }
        )

fit_started = time.perf_counter()
estimator = BayesBreakGaussian(k_max=8, regression_curve="mix_k").fit(coordinates, observations)
fit_seconds = time.perf_counter() - fit_started
predictions = estimator.predict(coordinates)
segment_labels = estimator.transform(coordinates)
k_map, map_boundaries, map_means = estimator.get_map_segmentation()

record_check("fit", "finite log evidence", lambda: bool(np.isfinite(estimator.log_evidence_)), True)
record_check("sum-product DP", "posterior k normalizes", lambda: bool(np.isclose(estimator.k_posterior_.sum(), 1.0, atol=1e-10)), True)
record_check("block evidence", "matrix shape", lambda: estimator.log_block_evidence_.shape, (111, 111))
record_check("block evidence", "admissibility equals finite mask", lambda: bool(np.array_equal(estimator.admissibility_mask_, np.isfinite(estimator.log_block_evidence_))), True)
record_check("max-sum DP", "strict boundary partition", lambda: map_boundaries[0] == 0 and map_boundaries[-1] == 110 and np.all(np.diff(map_boundaries) > 0), True)
record_check("prediction", "prediction shape", lambda: predictions.shape, observations.shape)
record_check("transform", "labels cover MAP segments", lambda: (segment_labels.min(), segment_labels.max()), (0, k_map - 1))
record_check("sklearn", "clone preserves k_max", lambda: clone(estimator).k_max, 8)
record_check("score", "training score is finite", lambda: bool(np.isfinite(estimator.score(coordinates, observations))), True)

verification_frame = pd.DataFrame(verification_records)
display(verification_frame)
print(f"fit_seconds={fit_seconds:.4f}, k_map={k_map}, boundaries={map_boundaries}")
assert (verification_frame.status == "pass").all()

## 7. Inspect posterior state and DP diagnostics

The four panels connect raw data to the stored computational objects. The evidence heatmap displays only finite admissible blocks. The diagnostic report independently checks posterior normalization, forward/backward duality, the boundary-event sum, and MAP backtracking consistency.

In [ ]:
diagnostic_report = run_dp_diagnostics(estimator)
diagnostic_frame = pd.DataFrame([check.to_dict() for check in diagnostic_report.checks])
display(diagnostic_frame)
assert diagnostic_report.passed

fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
ax = axes[0, 0]
ax.scatter(coordinates[:, 0], observations, s=12, color="#466365", alpha=0.6, label="observed")
ax.plot(coordinates[:, 0], estimator.map_curve_, color="#00798C", linewidth=2.2, label="MAP curve")
for boundary in true_boundaries[1:-1]:
    ax.axvline(boundary, color="#D1495B", linestyle="--", linewidth=1, label="true" if boundary == 35 else None)
for boundary in estimator.map_boundaries_[1:-1]:
    ax.axvline(boundary, color="#EDAE49", linestyle=":", linewidth=1.5, label="MAP" if boundary == estimator.map_boundaries_[1] else None)
ax.set(title="Signal and MAP segmentation", xlabel="index", ylabel="response")
ax.legend(ncol=2, fontsize=8)

ax = axes[0, 1]
k_values = np.arange(1, estimator.k_posterior_.size + 1)
ax.bar(k_values, estimator.k_posterior_, color="#00798C")
ax.axvline(estimator.k_map_, color="#D1495B", linestyle="--", label=f"MAP k={estimator.k_map_}")
ax.set(title="Posterior segment count", xlabel="number of segments k", ylabel="P(k | y)")
ax.legend()

ax = axes[1, 0]
positions = np.arange(1, estimator.n_)
ax.fill_between(positions, estimator.boundary_marginals_, color="#30638E", alpha=0.55)
for boundary in true_boundaries[1:-1]:
    ax.axvline(boundary, color="#D1495B", linestyle="--", linewidth=1)
ax.set(title=f"Boundary marginals conditional on k={estimator.k_map_}", xlabel="candidate boundary", ylabel="probability")

ax = axes[1, 1]
evidence = np.where(estimator.admissibility_mask_, estimator.log_block_evidence_, np.nan)
image = ax.imshow(evidence, origin="lower", cmap="viridis", aspect="auto")
ax.set(title="Finite block log evidence", xlabel="block stop", ylabel="block start")
fig.colorbar(image, ax=ax, label="log evidence")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "posterior_state.png", bbox_inches="tight")
plt.show()

## 8. Analyze prior sensitivity and a negative control

Prior variants are perturbations, not alternative truths. The shuffled control preserves all response values but destroys their sequence order. Comparing posterior summaries reveals sequence sensitivity without claiming the fitted shuffled partition is ground truth.

In [ ]:
sensitivity_report = run_prior_sensitivity(estimator)
sensitivity_frame = pd.DataFrame(sensitivity_report.extra["variants"])
display(sensitivity_frame)

shuffled_estimator = BayesBreakGaussian(k_max=8, regression_curve="mix_k").fit(
    coordinates, shuffled_observations
)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
variant_names = sensitivity_frame["variant"].str.replace("_", " ")
axes[0].barh(variant_names, sensitivity_frame["delta_pk_tv"], color="#00798C")
axes[0].set(title="Prior sensitivity: segment-count TV", xlabel="TV distance")
axes[1].barh(variant_names, sensitivity_frame["delta_bm_l1"], color="#EDAE49")
axes[1].set(title="Prior sensitivity: boundary change", xlabel="boundary-marginal L1")

axes[2].plot(k_values, estimator.k_posterior_, marker="o", label="ordered signal", color="#00798C")
axes[2].plot(k_values, shuffled_estimator.k_posterior_, marker="s", label="shuffled control", color="#D1495B")
axes[2].set(title="Order-sensitive posterior contrast", xlabel="number of segments k", ylabel="P(k | y)")
axes[2].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "sensitivity_and_control.png", bbox_inches="tight")
plt.show()

assert np.isclose(shuffled_estimator.k_posterior_.sum(), 1.0)
assert shuffled_estimator.n_ == estimator.n_

## 9–10. Measure and plot performance

Exact segmentation stores an $O(n^2)$ block-evidence table. This small benchmark uses three repeats per size and reports median runtime, interquartile range, throughput, and evidence-table memory. It is a local diagnostic, not a cross-machine benchmark.

In [ ]:
benchmark_rows = []
for sample_size in [40, 80, 120, 160]:
    local_rng = np.random.default_rng(SEED + sample_size)
    local_latent = np.r_[np.zeros(sample_size // 2), np.full(sample_size - sample_size // 2, 1.5)]
    local_y = local_latent + local_rng.normal(0.0, 0.5, sample_size)
    local_x = np.arange(sample_size).reshape(-1, 1)
    for repetition in range(3):
        started = time.perf_counter()
        fitted = BayesBreakGaussian(k_max=6).fit(local_x, local_y)
        elapsed = time.perf_counter() - started
        benchmark_rows.append(
            {
                "n": sample_size,
                "repetition": repetition,
                "runtime_seconds": elapsed,
                "throughput_obs_per_second": sample_size / elapsed,
                "evidence_memory_mib": fitted.log_block_evidence_.nbytes / 1024**2,
            }
        )
benchmark_frame = pd.DataFrame(benchmark_rows)
benchmark_summary = benchmark_frame.groupby("n", as_index=False).agg(
    runtime_median=("runtime_seconds", "median"),
    runtime_q25=("runtime_seconds", lambda values: values.quantile(0.25)),
    runtime_q75=("runtime_seconds", lambda values: values.quantile(0.75)),
    throughput_median=("throughput_obs_per_second", "median"),
    memory_mib=("evidence_memory_mib", "max"),
)
display(benchmark_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(benchmark_summary.n, benchmark_summary.runtime_median, marker="o", color="#00798C")
axes[0].fill_between(benchmark_summary.n, benchmark_summary.runtime_q25, benchmark_summary.runtime_q75, color="#00798C", alpha=0.2)
axes[0].set(title="Fit-time scaling", xlabel="observations n", ylabel="seconds (median and IQR)")
axes[1].plot(benchmark_summary.n, benchmark_summary.memory_mib, marker="s", color="#D1495B")
axes[1].set(title="Block-evidence storage", xlabel="observations n", ylabel="MiB")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "performance.png", bbox_inches="tight")
plt.show()

## 11. Analyze errors and edge cases

Expected failures are successful contract checks only when both exception type and message fragment match. This section covers an invalid segment budget, nonfinite observations, mismatched lengths, and prediction before fitting.

In [ ]:
edge_cases = [
    ("k_max=0", lambda: BayesBreakGaussian(k_max=0).fit(coordinates, observations), ValueError, "k_max"),
    ("nonfinite y", lambda: BayesBreakGaussian().fit(coordinates, np.where(np.arange(110) == 3, np.nan, observations)), ValueError, "finite"),
    ("length mismatch", lambda: BayesBreakGaussian().fit(coordinates[:-1], observations), ValueError, "sample dimension"),
    ("predict before fit", lambda: BayesBreakGaussian().predict(coordinates[:2]), RuntimeError, "fitted"),
]
edge_rows = []
for name, operation, expected_type, expected_message in edge_cases:
    started = time.perf_counter()
    try:
        operation()
        edge_rows.append({"case": name, "status": "fail", "exception": "none", "message": "", "elapsed_ms": 1000 * (time.perf_counter() - started)})
    except Exception as error:
        passed = isinstance(error, expected_type) and expected_message.lower() in str(error).lower()
        edge_rows.append(
            {
                "case": name,
                "status": "pass" if passed else "fail",
                "exception": type(error).__name__,
                "message": str(error),
                "elapsed_ms": 1000 * (time.perf_counter() - started),
            }
        )
edge_frame = pd.DataFrame(edge_rows)
display(edge_frame)
assert (edge_frame.status == "pass").all()

## 12. Generate the verification report

The final cell combines invariant checks, built-in diagnostics, and expected-failure contracts. It exports machine-readable tables and an at-a-glance status chart. Re-running the notebook replaces only files in its dedicated report directory.

In [ ]:
diagnostic_export = diagnostic_frame.assign(component="DP diagnostics").rename(columns={"name": "check"})
diagnostic_export["status"] = np.where(diagnostic_export.passed, "pass", "fail")
combined_status = pd.concat(
    [
        verification_frame[["component", "check", "status", "elapsed_ms"]],
        diagnostic_export[["component", "check", "status"]].assign(elapsed_ms=np.nan),
        edge_frame.rename(columns={"case": "check"}).assign(component="edge cases")[["component", "check", "status", "elapsed_ms"]],
    ],
    ignore_index=True,
)
status_counts = combined_status.status.value_counts().reindex(["pass", "fail", "error"], fill_value=0)

fig, ax = plt.subplots(figsize=(6.5, 3.6))
colors = ["#2A9D8F", "#E9C46A", "#D1495B"]
ax.bar(status_counts.index, status_counts.values, color=colors)
for index, value in enumerate(status_counts.values):
    ax.text(index, value + 0.1, str(value), ha="center", fontweight="bold")
ax.set(title="Core verification status", ylabel="number of checks", ylim=(0, max(status_counts.max() + 2, 3)))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "status.png", bbox_inches="tight")
plt.show()

verification_frame.to_csv(OUTPUT_DIR / "checks.csv", index=False)
diagnostic_frame.to_csv(OUTPUT_DIR / "dp_diagnostics.csv", index=False)
benchmark_frame.to_csv(OUTPUT_DIR / "benchmark_runs.csv", index=False)
benchmark_summary.to_csv(OUTPUT_DIR / "benchmark_summary.csv", index=False)
edge_frame.to_csv(OUTPUT_DIR / "edge_cases.csv", index=False)
report = {
    "environment": ENVIRONMENT,
    "fixture": {"n": estimator.n_, "true_boundaries": true_boundaries},
    "fit": {"seconds": fit_seconds, "k_map": estimator.k_map_, "boundaries": estimator.map_boundaries_},
    "status_counts": {key: int(value) for key, value in status_counts.items()},
    "all_required_checks_passed": bool((combined_status.status == "pass").all()),
}
(OUTPUT_DIR / "report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
assert report["all_required_checks_passed"]